### Copyright (C) Infineon Technologies AG 2025
 
Copyright (c) 2025, Infineon Technologies AG, or an affiliate of Infineon Technologies AG. All rights reserved.
This software, associated documentation and materials ("Software") is owned by Infineon Technologies AG or one of its affiliates ("Infineon") and is protected by and subject to worldwide patent protection, worldwide copyright laws, and international treaty provisions. Therefore, you may use this Software only as provided in the license agreement accompanying the software package from which you obtained this Software. If no license agreement applies, then any use, reproduction, modification, translation, or compilation of this Software is prohibited without the express written permission of Infineon.

Disclaimer: UNLESS OTHERWISE EXPRESSLY AGREED WITH INFINEON, THIS SOFTWARE IS PROVIDED AS-IS, WITH NO WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, ALL WARRANTIES OF NON-INFRINGEMENT OF THIRD-PARTY RIGHTS AND IMPLIED WARRANTIES SUCH AS WARRANTIES OF FITNESS FOR A SPECIFIC USE/PURPOSE OR MERCHANTABILITY. Infineon reserves the right to make changes to the Software without notice. You are responsible for properly designing, programming, and testing the functionality and safety of your intended application of the Software, as well as complying with any legal requirements related to its use. Infineon does not guarantee that the Software will be free from intrusion, data theft or loss, or other breaches ("Security Breaches"), and Infineon shall have no liability arising out of any Security Breaches. Unless otherwise explicitly approved by Infineon, the Software may not be used in any application where a failure of the Product or any consequences of the use thereof can reasonably be expected to result in personal injury.

# Post-Training Quantization: CNN for Rain-Drop Classification

This notebook trains a small CNN to classify clear versus raindrop images, then compares the floating-point model with static and dynamic post-training quantization (PTQ). It prepares calibration data, validates accuracy and output fidelity, converts the original and quantized models for embedded deployment, profiles their execution, and summarizes latency, model size, accuracy, and numerical error. The workflow demonstrates PTQ trade-offs rather than replacing a full model-development or production-validation process.

## Notebook Structure

1. **Import libraries and helper functions**
2. **Prepare the Raindrop Clarity data and inputs**
3. **Build and train the original CNN**
4. **Generate representative inputs and export the FP32 model**
5. **Convert the original FP32 model**
6. **Prepare PTQ variants and calibration data**
7. **Apply static post-training quantization**
8. **Apply dynamic post-training quantization**
9. **Validate floating-point and quantized models**
10. **Convert the quantized models**
11. **Compare per-node profiling results**
12. **Compare latency, model size, accuracy, and output fidelity**
13. **References and Generated Artifacts**

## 1. Import libraries and helper functions

Import PyTorch, dataset/model helpers, quantization utilities, and the conversion client used by the workflow.

In [ ]:
import os
import sys
from torchsummary import summary
import torch.optim as optim

import modelling_helper as mh

parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from _CentralScripts.python_flask_client import CallTools
import _CentralScripts.helper_functions as cs
import _CentralScripts.mobilenet_helper as mo
import _CentralScripts.quantization_helper as qh
import MobileNetV3ClassificationRainDrops.modelling_helper as mm

device = cs.get_device()

## 2. Prepare the Raindrop Clarity data and inputs

The notebook downloads and restructures the [Raindrop Clarity dataset](https://github.com/jinyeying/RaindropClarity?tab=readme-ov-file) [2]. The source dataset was designed for algorithmic raindrop removal, but this example repurposes its images for binary classification of clear and raindrop conditions. The helper creates train, validation, and test splits and applies the configured 90-pixel resolution and preprocessing.

Training data are used to fit the CNN, validation data are used for model comparison, and representative tensors are used for export/conversion checks. Because the source data target raindrop removal rather than classification, class balance, camera variation, weather coverage, and the resulting classification labels should be checked before deployment.

In [ ]:
batch_size = 16
resolution = 90
SEED = 42

path = mm.fetch_sort_data()
dataloader, class_names, dataset_sizes = mo.get_dataloader(
    os.path.join(path, "train"),
    batch_size,
    mode="train",
    resolution=resolution,
    SEED=SEED,
)
_, input_size = mm.get_input_dataloader(dataloader, is_plot=True)

## 3. Build and train the original CNN

Create a two-class CNN for clear-versus-raindrop classification and inspect its input/output shape before training.

In [ ]:
import torch.nn as nn

model = mh.get_model(2)
summary(model, input_size, device="cpu")

### Train the floating-point baseline

Train the original FP32 CNN with cross-entropy loss and Adam. The one-epoch run is a compact demonstration; production comparisons should record the training configuration and use a representative training schedule.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
mh.train_model(model, dataloader, dataset_sizes, criterion, optimizer, num_epochs=1)

## 4. Generate representative inputs and export the FP32 model

Select representative tensors and compute their floating-point outputs. The saved input/output pair defines the conversion validation contract for the original and quantized models.

In [ ]:
input_target, other_input_target, _ = mm.get_input_dataloader(
    dataloader, is_plot=True, is_multiple_inputs=True, is_code_example=True
)
output_target = cs.get_predictions("torch", model, input_target)
other_output_target = cs.get_predictions("torch", model, other_input_target)
mm.save_output_c_header(output_target, other_output_target)

The model is exported as `cnn_weather` together with representative input and expected output artifacts. These files are the baseline for quantization and embedded conversion.

In [ ]:
model_name = "cnn_weather"
cs.save_all(model_name, input_target, output_target, model, origin="torch")

## 5. Convert the original FP32 model

Convert and profile the floating-point baseline before quantization. The deployment targets are AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4.

### Prepare the conversion service

Ensure that the Docker container containing the model-conversion toolchain is available.

In [ ]:
cs.ensure_docker_container()

### Convert the floating-point baseline

Convert the exported FP32 model for the selected target and retain its timing/profile results as the reference for quantized variants.

In [ ]:
model_folder, onnx_model_file = cs.get_output_paths(model_name)

target = "tc4dx_ppu"  # Available targets: "tc3xx", "tc4dx", "tc4dx_ppu", "arm_m4"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
    profile=True,
)
tool.convert_model()

## 6. Prepare PTQ variants and calibration data

Create output folders for the preprocessed, static-int8, and dynamic-uint8 variants. ONNX Runtime QDQ transformations [3] are applied after the FP32 model has been exported. Calibration data represent the preprocessed training-domain inputs and are used only by static quantization to estimate activation ranges.

In [ ]:
model_folder, onnx_model_file = cs.get_output_paths(model_name)

model_paths, options = mh.create_folders_quantization(model_name)

(
    model_path_preprocessed,
    model_path_ptq_int8,
    model_path_dynamic_ptq,
) = model_paths

## 7. Apply static post-training quantization

Static PTQ quantizes weights and activations using calibration ranges. It can reduce memory traffic for CNNs, but its accuracy and output fidelity depend on calibration data that represent deployment inputs [4].

In [ ]:
qh.onnx_preprocessing(onnx_model_file, model_path_preprocessed)

### Prepare calibration data

Select 100 preprocessed samples from the training data and identify the ONNX input name. These samples calibrate activation ranges; they are not an additional evaluation split.

In [ ]:
calib_samples, _ = mh.get_calib_data(path, resolution=resolution, num_samples=100)
input_name = mh.get_input_name(model_path_preprocessed)

### Quantize with signed INT8

Apply static signed-INT8 quantization and retain the generated model for validation, conversion, timing, and comparison against the FP32 baseline.

In [ ]:
from onnxruntime.quantization import QuantType

# int 8 quantization
qh.static_quantization(
    model_path_preprocessed,
    model_path_ptq_int8,
    calib_samples,
    input_name,
    type=QuantType.QInt8,
    input_target=input_target,
)

## 8. Apply dynamic post-training quantization

Dynamic PTQ quantizes weights ahead of time while determining activation quantization parameters at runtime. It is often useful for compute- or memory-bound recurrent models, but this notebook intentionally measures the trade-offs on a CNN [4].

In [ ]:
qh.dynamic_quantization(
    model_path_preprocessed,
    model_path_dynamic_ptq,
    type=QuantType.QUInt8,
    input_target=input_target,
)

## 9. Validate floating-point and quantized models

Evaluate the FP32, static-INT8, and dynamic-quantized variants on the held-out validation split. Compare classification accuracy and compiled output fidelity, including maximum absolute error and RMSE where available, before interpreting timing or model-size changes.

In [ ]:
dataloader_val, class_names, dataset_sizes = mo.get_dataloader(
    os.path.join(path, "val"), batch_size=1, mode="val", resolution=resolution
)

model_paths.insert(0, onnx_model_file)
accuracies = qh.validate_quantization_options((model_paths), dataloader_val)

## 10. Convert the quantized models

Convert each quantized variant with the same selected target and profiling configuration used for the FP32 baseline. The generated results enable an apples-to-apples deployment comparison across AURIX™ and TRAVEO™ targets.

### Generate quantized deployment artifacts

Compile the static- and dynamic-quantized models, then prepare the code example inputs when requested. The conversion outputs include target-specific source, binaries, logs, timing, and profile data.

In [ ]:
for option in options:

    if option == "preprocessed":
        continue

    model_name_quantized = f"{model_name}_{option}"
    print(f"Processing model: {model_name_quantized}")
    model_folder, onnx_model_file = cs.get_output_paths(model_name_quantized)

    tool = CallTools(
        folder=model_folder,
        url="http://localhost:8080/convert",
        target=target,
        profile=True,
    )
    tool.convert_model()

In [ ]:
is_prepare_code_example = True

if is_prepare_code_example:
    mh.prepare_code_example()

## 11. Compare per-node profiling results

Compare estimated cycles per ONNX node across the FP32, static-quantized, and dynamic-quantized variants. Quantize/DequantizeLinear nodes are reported as static-quantization overhead, so the profile helps explain total latency changes.

In [ ]:
cs.plot_node_profile_compare(model_name, target=target, metric="estimated_cycles")

## 12. Compare latency, model size, accuracy, and output fidelity

The summary table compares the FP32 baseline with each PTQ method. Latency and estimated cycles come from `results.json`, compiled weights and activation buffers come from `model.md`, and accuracy comes from validation. Maximum absolute error and RMSE describe compiled-output fidelity relative to the ONNX reference.

In [ ]:
df = cs.compare_quantization_overview(model_name, target=target, accuracy=accuracies)

## 13. References and Generated Artifacts

- **Quantization context:** [PyTorch Quantization-Aware Training](https://pytorch.org/blog/quantization-aware-training/) [1].
- **Dataset:** [Raindrop Clarity](https://github.com/jinyeying/RaindropClarity?tab=readme-ov-file), Jin et al. (2024), originally created for day/night raindrop removal and repurposed here for clear-versus-raindrop classification [2].
- **Quantization references:** [ONNX Runtime quantization](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html) [3] and [PyTorch quantization introduction](https://pytorch.org/blog/introduction-to-quantization-on-pytorch) [4].
- **Generated model artifacts:** `cnn_weather` FP32 ONNX model, static-INT8 and dynamic-quantized models, representative inputs/outputs, and calibration data.
- **Deployment artifacts:** target-specific source, binaries, logs, timing, profiling, and comparison reports under `out/cnn_weather/` and `out/cnn_weather_<quantization-option>/test_cnn_weather_<quantization-option>/<target>/`.
- **Deployment targets:** AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4.
- **Interpretation:** compare accuracy, latency, model size, output fidelity, and conversion viability together; quantization gains are not meaningful if the validation or calibration data do not represent deployment conditions.